# 03. Advanced RAG: Hybrid Search, Reranking & Evaluation

**Topics covered:** Hybrid Search · Reranking · RAG Evaluation

This notebook builds on [02_basic_rag_pipeline.ipynb](https://github.com/S33mi/modern-ai-llm-journey/blob/main/04_rag_systems/02_basic_rag_pipeline.ipynb).

We will:
1. Combine **dense** (embeddings) and **sparse** (BM25) retrieval → **hybrid search**
2. Apply a **cross-encoder reranker** to improve ranking
3. Measure quality with simple **RAG evaluation** metrics
4. Keep everything **CPU/GPU compatible**

> **Note:** Transformers v5 removed the `"text2text-generation"` pipeline task.  
> Generation here uses `AutoModelForSeq2SeqLM.generate()` directly so it works on recent versions.

## 1. Setup

```bash
pip install transformers sentence-transformers faiss-cpu rank-bm25 scikit-learn numpy
```

In [2]:
# ! pip install transformers sentence-transformers faiss-cpu rank-bm25 scikit-learn numpy
# pip install transformers sentence-transformers faiss-gpu rank-bm25 scikit-learn numpy

In [3]:
import re
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import faiss
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cpu


## 2. Knowledge Base & Chunks

In [4]:
documents = [
    {
        "id": "doc1",
        "title": "Attention Mechanisms",
        "text": (
            "Scaled dot-product attention computes similarity between queries and keys, "
            "scales by the square root of the key dimension, applies softmax, and uses "
            "the resulting weights to combine values. Multi-head attention runs several "
            "attention heads in parallel so the model can capture different types of "
            "relationships. Causal masking prevents tokens from attending to future "
            "positions, which is essential for autoregressive language models like GPT."
        ),
    },
    {
        "id": "doc2",
        "title": "Transformers and Residuals",
        "text": (
            "A Transformer block typically contains multi-head self-attention followed by "
            "a position-wise feed-forward network. Residual connections and layer "
            "normalization stabilize training of deep stacks. Pre-LN is the dominant design "
            "in modern LLMs such as GPT-2, LLaMA, and Mistral. The feed-forward network "
            "usually expands the hidden size by a factor of four and uses GELU or SwiGLU."
        ),
    },
    {
        "id": "doc3",
        "title": "Fine-Tuning and LoRA",
        "text": (
            "Full fine-tuning updates every parameter of a pre-trained model. This is "
            "expensive in memory and storage. LoRA freezes the base weights and injects "
            "small trainable low-rank matrices. The rank r controls capacity; alpha scales "
            "the update. QLoRA combines 4-bit quantization of the base model with LoRA "
            "adapters so that 7B to 13B models can be fine-tuned on a single consumer GPU."
        ),
    },
    {
        "id": "doc4",
        "title": "Embeddings and Similarity",
        "text": (
            "Sentence embeddings map text into a dense vector space where semantically "
            "similar sentences lie close together. Cosine similarity is the standard "
            "metric. Models such as all-MiniLM-L6-v2 and bge-small produce strong "
            "embeddings for retrieval. Mean pooling of the last hidden states is a common "
            "way to obtain a fixed-size sentence vector from a Transformer."
        ),
    },
    {
        "id": "doc5",
        "title": "RAG Overview",
        "text": (
            "Retrieval-Augmented Generation (RAG) combines a retriever with a generator. "
            "Documents are chunked and embedded into a vector store. At query time the "
            "system retrieves the most relevant chunks and passes them as context to an "
            "LLM, which produces an answer grounded in that context. Hybrid search and "
            "reranking often improve retrieval quality before generation."
        ),
    },
]

# Simple sentence-ish chunks (keep short for demo)
chunks = []
for doc in documents:
    # split on periods while keeping content
    parts = [p.strip() + "." for p in doc["text"].split(".") if p.strip()]
    for i, part in enumerate(parts):
        chunks.append({
            "chunk_id": f"{doc['id']}_c{i}",
            "doc_id": doc["id"],
            "title": doc["title"],
            "text": part,
        })

print(f"{len(chunks)} chunks")
for c in chunks[:3]:
    print(f"  {c['chunk_id']}: {c['text'][:70]}...")

20 chunks
  doc1_c0: Scaled dot-product attention computes similarity between queries and k...
  doc1_c1: Multi-head attention runs several attention heads in parallel so the m...
  doc1_c2: Causal masking prevents tokens from attending to future positions, whi...


## 3. Dense Retriever (FAISS)

In [5]:
embed_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=DEVICE,
)

chunk_texts = [c["text"] for c in chunks]
dense_emb = embed_model.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=False)
dense_emb = np.asarray(dense_emb, dtype="float32")

faiss_index = faiss.IndexFlatIP(dense_emb.shape[1])
faiss_index.add(dense_emb)

def dense_search(query: str, k: int = 5):
    q = embed_model.encode([query], normalize_embeddings=True)
    scores, idxs = faiss_index.search(np.asarray(q, dtype="float32"), k)
    return [
        {"chunk": chunks[int(i)], "score": float(s), "source": "dense"}
        for s, i in zip(scores[0], idxs[0])
    ]

print("Dense index ready:", faiss_index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dense index ready: 20


## 4. Sparse Retriever (BM25)

BM25 is a classic lexical ranking function. It catches exact keywords that dense models sometimes miss.

In [6]:
def tokenize(text: str):
    return re.findall(r"[a-z0-9]+", text.lower())

tokenized_corpus = [tokenize(t) for t in chunk_texts]
bm25 = BM25Okapi(tokenized_corpus)

def sparse_search(query: str, k: int = 5):
    tokens = tokenize(query)
    scores = bm25.get_scores(tokens)
    top = np.argsort(scores)[::-1][:k]
    return [
        {"chunk": chunks[int(i)], "score": float(scores[i]), "source": "sparse"}
        for i in top if scores[i] > 0
    ]

print("BM25 ready.")
print([r["chunk"]["text"][:50] for r in sparse_search("LoRA rank quantization", k=3)])

BM25 ready.
['LoRA freezes the base weights and injects small tr', 'QLoRA combines 4-bit quantization of the base mode', 'The rank r controls capacity; alpha scales the upd']


## 5. Hybrid Search (Dense + Sparse)

Common fusion methods:

- **RRF (Reciprocal Rank Fusion)** – robust, no score calibration needed
- Weighted sum of normalized scores

We use **RRF**:

In [7]:
def reciprocal_rank_fusion(result_lists, k_rrf: int = 60):
    """
    result_lists: list of ranked lists, each item has key 'chunk' with 'chunk_id'
    """
    scores = {}
    payload = {}
    for results in result_lists:
        for rank, item in enumerate(results):
            cid = item["chunk"]["chunk_id"]
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k_rrf + rank + 1)
            payload[cid] = item["chunk"]
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [{"chunk": payload[cid], "score": sc, "source": "hybrid"} for cid, sc in ranked]


def hybrid_search(query: str, k: int = 5):
    dense_hits = dense_search(query, k=k)
    sparse_hits = sparse_search(query, k=k)
    fused = reciprocal_rank_fusion([dense_hits, sparse_hits])
    return fused[:k]


q = "How does QLoRA save memory?"
print("Dense:", [h["chunk"]["title"] for h in dense_search(q, 3)])
print("Sparse:", [h["chunk"]["title"] for h in sparse_search(q, 3)])
print("Hybrid:", [h["chunk"]["title"] for h in hybrid_search(q, 3)])

Dense: ['Fine-Tuning and LoRA', 'Fine-Tuning and LoRA', 'RAG Overview']
Sparse: ['Fine-Tuning and LoRA', 'Fine-Tuning and LoRA']
Hybrid: ['Fine-Tuning and LoRA', 'Fine-Tuning and LoRA', 'RAG Overview']


## 6. Cross-Encoder Reranking

Bi-encoders (sentence-transformers) encode query and doc **separately** → fast, approximate.  
Cross-encoders score **(query, doc)** pairs jointly → slower, usually more accurate.

Typical pattern: retrieve top-20 with hybrid, **rerank** to top-3–5.

In [8]:
# Small cross-encoder that runs on CPU
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)

def rerank(query: str, hits: list, top_n: int = 3):
    if not hits:
        return []
    pairs = [(query, h["chunk"]["text"]) for h in hits]
    scores = reranker.predict(pairs)
    order = np.argsort(scores)[::-1][:top_n]
    out = []
    for i in order:
        item = hits[int(i)].copy()
        item["rerank_score"] = float(scores[int(i)])
        item["source"] = "reranked"
        out.append(item)
    return out


raw = hybrid_search(q, k=6)
reranked = rerank(q, raw, top_n=3)
print("After rerank:")
for h in reranked:
    print(f"  [{h['rerank_score']:.3f}] {h['chunk']['title']}: {h['chunk']['text'][:80]}...")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

After rerank:
  [-0.460] Fine-Tuning and LoRA: QLoRA combines 4-bit quantization of the base model with LoRA adapters so that 7...
  [-10.275] Fine-Tuning and LoRA: This is expensive in memory and storage....
  [-11.315] RAG Overview: At query time the system retrieves the most relevant chunks and passes them as c...


## 7. Full Retrieve → Rerank → Generate

Uses `AutoModelForSeq2SeqLM.generate()` (Transformers v5 compatible).

In [9]:
GEN_NAME = "google/flan-t5-base" if DEVICE == "cuda" else "google/flan-t5-small"
gen_tok = AutoTokenizer.from_pretrained(GEN_NAME)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(GEN_NAME).to(DEVICE)
gen_model.eval()

def generate_answer(prompt: str, max_new_tokens: int = 100) -> str:
    inputs = gen_tok(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    return gen_tok.decode(out[0], skip_special_tokens=True).strip()


def build_prompt(query: str, hits: list) -> str:
    ctx = "\n\n".join(
        f"[{i}] ({h['chunk']['title']}) {h['chunk']['text']}"
        for i, h in enumerate(hits, 1)
    )
    return (
        "Use only the context to answer. "
        "If the answer is not in the context, say I don't know.\n\n"
        f"Context:\n{ctx}\n\nQuestion: {query}\nAnswer:"
    )


def advanced_rag(query: str, retrieve_k: int = 8, rerank_n: int = 3) -> dict:
    hybrid_hits = hybrid_search(query, k=retrieve_k)
    top = rerank(query, hybrid_hits, top_n=rerank_n)
    prompt = build_prompt(query, top)
    answer = generate_answer(prompt)
    return {"answer": answer, "contexts": top}


result = advanced_rag("What is multi-head attention?")
print("Answer:", result["answer"])
print("\nContexts used:")
for h in result["contexts"]:
    print(f"  - {h['chunk']['title']}: {h['chunk']['text'][:70]}...")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Answer: [1]

Contexts used:
  - Attention Mechanisms: Multi-head attention runs several attention heads in parallel so the m...
  - Transformers and Residuals: A Transformer block typically contains multi-head self-attention follo...
  - Attention Mechanisms: Scaled dot-product attention computes similarity between queries and k...


## 8. RAG Evaluation (Lightweight)

For a serious system you would use datasets like RAGAS, HotpotQA, or custom labeled sets.  
Here we define a tiny gold set and compute:

- **Retrieval hit rate**: is any gold doc among top-k?
- **Answer contains keyword**: crude but useful for demos

In [10]:
eval_set = [
    {
        "question": "What does causal masking do?",
        "gold_doc_ids": ["doc1"],
        "must_include": ["future"],
    },
    {
        "question": "What is LoRA?",
        "gold_doc_ids": ["doc3"],
        "must_include": ["low-rank"],
    },
    {
        "question": "How are sentence embeddings typically pooled?",
        "gold_doc_ids": ["doc4"],
        "must_include": ["mean"],
    },
    {
        "question": "What is hybrid search used for in RAG?",
        "gold_doc_ids": ["doc5"],
        "must_include": ["retriev"],
    },
]

def evaluate_rag(eval_set, retrieve_k=8, rerank_n=3):
    hits = 0
    keyword_ok = 0
    for ex in eval_set:
        out = advanced_rag(ex["question"], retrieve_k=retrieve_k, rerank_n=rerank_n)
        retrieved_docs = {h["chunk"]["doc_id"] for h in out["contexts"]}
        if retrieved_docs & set(ex["gold_doc_ids"]):
            hits += 1
        ans_l = out["answer"].lower()
        if any(k.lower() in ans_l for k in ex["must_include"]):
            keyword_ok += 1
        print(f"Q: {ex['question']}")
        print(f"   A: {out['answer'][:120]}")
        print(f"   docs={retrieved_docs} gold={ex['gold_doc_ids']}\n")
    n = len(eval_set)
    return {
        "retrieval_hit_rate": hits / n,
        "keyword_match_rate": keyword_ok / n,
        "n": n,
    }


metrics = evaluate_rag(eval_set)
print("Metrics:", metrics)

Q: What does causal masking do?
   A: prevents tokens from attending to future positions
   docs={'doc5', 'doc1', 'doc4'} gold=['doc1']

Q: What is LoRA?
   A: [1]
   docs={'doc3', 'doc4'} gold=['doc3']

Q: How are sentence embeddings typically pooled?
   A: [3]
   docs={'doc4'} gold=['doc4']

Q: What is hybrid search used for in RAG?
   A: improve retrieval quality before generation
   docs={'doc2', 'doc5'} gold=['doc5']

Metrics: {'retrieval_hit_rate': 1.0, 'keyword_match_rate': 0.5, 'n': 4}


**retrieval_hit_rate = 1.0**

The retriever found the correct document in all 4 questions. Retrieval is working well.

**keyword_match_rate = 0.5**

Only 2 out of 4 answers contained the required keywords.

**The generator is weak** — it sometimes produces garbage answers like [1] or [3] instead of proper explanations.

Q: What is LoRA?

Retrieved correct doc, but answer is useless ([1]) and missing keyword “low-rank”

Q: How are sentence embeddings typically pooled?

Retrieved correct doc, but answer is just [3] with keyword not found (“mean”)

## 9. Compare Pipelines Side by Side

In [11]:
def show_ranking(query: str):
    print(f"Query: {query}\n")
    print("Dense only:")
    for h in dense_search(query, 3):
        print(f"  {h['score']:.3f}  {h['chunk']['title']}")
    print("\nHybrid:")
    for h in hybrid_search(query, 3):
        print(f"  {h['score']:.4f}  {h['chunk']['title']}")
    print("\nHybrid + rerank:")
    for h in rerank(query, hybrid_search(query, 6), 3):
        print(f"  {h['rerank_score']:.3f}  {h['chunk']['title']}")


show_ranking("Why use residual connections in transformers?")

Query: Why use residual connections in transformers?

Dense only:
  0.377  Transformers and Residuals
  0.352  Transformers and Residuals
  0.179  Embeddings and Similarity

Hybrid:
  0.0328  Transformers and Residuals
  0.0161  Transformers and Residuals
  0.0161  Fine-Tuning and LoRA

Hybrid + rerank:
  0.412  Transformers and Residuals
  -9.196  Transformers and Residuals
  -9.376  Embeddings and Similarity


## 10. Summary

| Technique | Role |
|-----------|------|
| **Dense retrieval** | Semantic similarity (embeddings + FAISS) |
| **Sparse retrieval (BM25)** | Keyword / lexical match |
| **Hybrid (RRF)** | Fuse ranked lists without score calibration |
| **Cross-encoder rerank** | Re-score top candidates more accurately |
| **Evaluation** | Hit rate on gold docs + simple answer checks |

### Recommended production pattern

```text
query
  → hybrid retrieve (dense + BM25, top 20–50)
  → cross-encoder rerank (top 3–5)
  → prompt with context
  → generator (Seq2Seq or Causal LM)
```

### Transformers v5 note

Avoid `pipeline("text2text-generation", ...)`. Prefer:

```python
model = AutoModelForSeq2SeqLM.from_pretrained(...)
inputs = tokenizer(prompt, return_tensors="pt")
ids = model.generate(**inputs, max_new_tokens=100)
text = tokenizer.decode(ids[0], skip_special_tokens=True)
```

---

**You have completed the `04_rag_systems` [series](https://github.com/S33mi/modern-ai-llm-journey/blob/main/04_rag_systems).**

Next folder: **`05_agents`**  
→ [`01_react_agent_from_scratch.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/blob/main/05_agents/)

---
**For contribution and insihght:** [**S33mi**](https://github.com/S33mi)

Open to Data Science/Analytics and ML/AI related opportunities